# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Iqra411/lyrank-ML-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.* RANKED ACTIONS: queue built from the Week-5/6 Random Forest's decline_probability
(trained on full data for deployment). Action mix across 30,000 scored pages:
- monitor: 17,009 (56.7%)
- refresh: 9,569 (31.9%)
- refresh_and_review_ctr: 3,340 (11.1%) -- declining risk AND ctr below its
  position-tier average, likely a title/snippet problem layered on top
- expand_and_refresh: 82 (0.3%) -- thin (<1,200 words) but already earning
  meaningful impressions (>=250)

ARCHETYPE -> ACTION MAPPING (freshness_tier x position_tier cells, n>=50 only,
per training-honest-models' minimum-bucket-size convention):

| Archetype (freshness x position) | n | decline rate | avg impressions | Suggested action |
|---|---|---|---|---|
| 91-180 x top_3 | 302 | 67.9% | 9,210 | refresh -- highest decline rate on the highest-value pages |
| 91-180 x page_1 | 3,335 | 62.3% | 10,005 | refresh -- largest n at this risk level, biggest queue segment |
| 91-180 x striking | 2,338 | 63.7% | 4,048 | refresh_and_review_ctr -- striking distance is the paper's own "highest ROI zone" |
| 0-30 x top_3 | 1,990 | 17.5% | 2,135 | monitor -- fresh AND well-positioned, lowest risk archetype measured |
| 91-180 x deep | 312 | 35.9% | 1,505 | monitor -- low value even if refreshed (deep position, low traffic) |

Observed pattern: decline rate is NOT simply "the older the worse" -- position
interacts with freshness. A stale top_3 page (67.9% decline) is far riskier
than a stale deep page (35.9%), even though both are equally "stale" by the
freshness_tier definition. This is why the queue ranks by the model's joint
probability rather than either dimension alone -- matches the Week-4 signal
audit's own MIXED verdict on freshness in isolation.

DECAY/REFRESH INSIGHT (decline rate by days-since-last-update, n>=30 bins):
0-30 days: 51.1% | 91-120 days: 61.3% (highest reliable point) | 181-270 days:
45.3%. The clearest, best-supported window is 0-30 -> 91-120 days: decline
risk climbs from 51.1% to 61.3% as pages go unrefreshed through roughly three
months. Bins beyond 120 days have too few rows (n<40) to trust directionally
-- observed only within this window, not claimed beyond it.

COST/VALUE THINKING: reviewing a page costs roughly the same reviewer-minutes
regardless of its traffic, so the return on that time is proportional to
impressions_90d. Ranking by decline_probability alone would put some
low-traffic pages ahead of high-traffic ones with similar risk. The queue
therefore reports decline_probability AND impressions_90d side by side (not
multiplied into one score, to keep the ranking auditable) -- a reviewer with
limited time should scan top-ranked HIGH-impression rows first, since a
correct refresh there returns more visible impact per minute spent than the
same fix on a 200-impression page.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
df = pd.read_csv('https://raw.githubusercontent.com/Iqra411/lyrank-ML-internship/main/data/raw/content_refresh_anonymized.csv')

numeric_fill_zero = [
    "search_volume","competition","cpc","word_count","char_count",
    "impressions_90d","clicks_90d","pageviews_90d","sessions_90d","users_90d",
    "engaged_sessions_90d","ai_sessions_90d","scroll_events_90d",
    "days_with_impressions","days_with_sessions","impressions_last_30d",
    "clicks_last_30d","sessions_last_30d","impressions_prev_30d",
    "clicks_prev_30d","sessions_prev_30d","content_age_days","age_tier_order",
    "days_since_last_update","ctr","avg_position","engagement_rate",
    "scroll_rate","ai_traffic_pct","trend_pct",
]
for c in numeric_fill_zero:
    df[c] = pd.to_numeric(df[c], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
cat_cols = ["competition_level","content_type","main_intent","provider_used","model_used",
            "age_tier","freshness_tier","word_count_tier","char_count_tier",
            "impression_tier","position_tier","trend_direction"]
for c in cat_cols:
    df[c] = df[c].fillna("unknown").astype(str).replace({"": "unknown", "nan": "unknown"})
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

MODEL_NUMERIC_FEATURES = [
    "search_volume","competition","cpc","word_count","char_count",
    "log_impressions_90d","log_clicks_90d","log_sessions_90d","log_ai_sessions_90d",
    "days_with_impressions","days_with_sessions","content_age_days",
    "days_since_last_update","ctr","avg_position","engagement_rate",
    "scroll_rate","ai_traffic_pct",
]
MODEL_CATEGORICAL_FEATURES = [
    "competition_level","content_type","main_intent","age_tier",
    "freshness_tier","word_count_tier","impression_tier","position_tier",
]
num_frame = df[MODEL_NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf,-np.inf], np.nan).fillna(0)
cat_frame = pd.get_dummies(df[MODEL_CATEGORICAL_FEATURES].astype(str), prefix=MODEL_CATEGORICAL_FEATURES, dtype=float)
feat = pd.concat([num_frame.reset_index(drop=True), cat_frame.reset_index(drop=True)], axis=1)
target = df["is_declining_label"].astype(int)

rf_full = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)
rf_full.fit(feat, target)
df["decline_probability"] = rf_full.predict_proba(feat)[:, 1]

def reason_codes(row):
    reasons = []
    if row["decline_probability"] >= 0.65:
        reasons.append("model_high_decline_risk")
    if row["freshness_tier"] == "91-180" and row["impressions_90d"] >= 300:
        reasons.append("stale_but_visible")
    if row["word_count"] > 0 and row["word_count"] < 1200 and row["impressions_90d"] >= 250:
        reasons.append("thin_visible_page")
    if row["position_tier"] in ("page_1", "top_3") and row["decline_probability"] >= 0.5:
        reasons.append("page_one_decay_risk")
    if row["impressions_90d"] >= 500 and 0 < row["avg_position"] <= 20 and row["ctr"] < 0.5:
        reasons.append("low_ctr_visible_page")
    return "|".join(reasons) if reasons else "general_review"

df["reason_codes"] = df.apply(reason_codes, axis=1)

def action(row):
    reasons = set(row["reason_codes"].split("|"))
    if "thin_visible_page" in reasons:
        return "expand_and_refresh"
    if "low_ctr_visible_page" in reasons and "model_high_decline_risk" in reasons:
        return "refresh_and_review_ctr"
    if "model_high_decline_risk" in reasons or "stale_but_visible" in reasons:
        return "refresh"
    return "monitor"

df["suggested_action"] = df.apply(action, axis=1)
df["rank"] = df["decline_probability"].rank(method="first", ascending=False).astype(int)

print("Action mix:\n", df["suggested_action"].value_counts())

# Archetype -> action table
arch = df.groupby(["freshness_tier","position_tier"], observed=True).agg(
    n=("content_id","size"), decline_rate=("is_declining_label","mean"),
    mean_impressions=("impressions_90d","mean"), mean_ctr=("ctr","mean"),
).reset_index()
arch = arch[arch["n"] >= 50].sort_values("decline_rate", ascending=False)
print("\nArchetype table:\n", arch.to_string(index=False))

# Decay/refresh curve
bins = [0,30,60,90,120,150,180,270,365,10000]
labels = ["0-30","31-60","61-90","91-120","121-150","151-180","181-270","271-365","365+"]
df["freshness_bin"] = pd.cut(df["days_since_last_update"], bins=bins, labels=labels, right=True)
decay = df.groupby("freshness_bin", observed=True).agg(n=("content_id","size"), decline_rate=("is_declining_label","mean")).reset_index()
print("\nDecay curve (n>=30 only):\n", decay[decay["n"] >= 30].to_string(index=False))

queue = df.sort_values("rank")[["content_id","client_id","rank","decline_probability","reason_codes",
    "suggested_action","impressions_90d","avg_position","ctr","word_count","freshness_tier","position_tier"]]

Action mix:
 suggested_action
monitor                   17009
refresh                    9569
refresh_and_review_ctr     3340
expand_and_refresh           82
Name: count, dtype: int64

Archetype table:
 freshness_tier position_tier    n  decline_rate  mean_impressions  mean_ctr
        91-180         top_3  302      0.678808       9209.745033  0.381258
         31-90      page_3_5   53      0.660377      11307.754717  0.084340
        91-180      striking 2338      0.636869       4047.524808  0.295000
        91-180        page_1 3335      0.623088      10005.439580  0.287757
          0-30      striking 4862      0.597491       2742.390991  0.336024
        91-180      page_3_5 2884      0.596394       7828.743412  0.137302
         31-90        page_1   53      0.566038       8189.245283  0.184717
          0-30        page_1 8342      0.549029       6684.987893  0.765621
         31-90      striking   67      0.537313       1543.701493  0.094179
          0-30      page_3_5 4283    

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*
Who uses this: a content team lead doing weekly refresh triage on an
already-existing content library. Not for: automated publishing
decisions, revenue forecasting, or any claim about what Google's
algorithm will do next.

Where it stops being valid:
- Any content outside this dataset's population filter (impressions_90d
  > 0, content_age_days >= 90) -- brand-new or zero-traffic pages were
  never scored
- Any client not resembling the 32 in this dataset -- the grouped-split
  test in w06 showed real performance variance across held-out clients
  (precision@50 measured at 0.74 on this specific 6-client holdout; a
  different holdout could differ)
- decline_probability is a risk estimate for prioritization, not a
  guarantee -- per w06's claim rewrite, this is decision-support, not
  a prediction

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*
Before acting on ANY row, a human should check:
1. Is the decline external (competitor, SERP feature change, algorithm
   update) rather than content quality? The model can't distinguish
   these -- flagged in the Week-4 top-10 review as a real "wrong if" case.
2. Does refreshing make business sense for this specific client/page
   right now (seasonality, ongoing campaigns, planned deprecation)?
3. For refresh_and_review_ctr rows: check the actual title/snippet
   before assuming a content refresh (not a CTR fix) is the right move.

No-go list -- never automate:
- Auto-publishing any content change without human review
- Using this queue as an input to a client-facing report without
  re-stating it in decision-support language
- Treating model_high_decline_risk alone (no other reason code) as
  proof of a content quality problem -- it's the least explainable
  reason code and needs the most manual verification

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*
Signals the recommendations have gone stale:
- Precision@50 on a fresh grouped holdout drops meaningfully below the
  0.74 measured in w05/w06 -- retrain
- The 32-client training population no longer represents the active
  client base (new clients added, old ones churned)
- Base decline rate drifts far from 54.2% (a shift in what "normal"
  looks like across the portfolio) -- re-check the label definition
  and feature distributions before trusting the same thresholds
- Any of the leakage-audit checks from w06 would need re-running if the
  underlying data pipeline changes (new columns, new export logic)

  Add: if the portfolio's impression distribution shifts heavily (e.g. traffic
consolidates onto far fewer pages), the cost/value ranking logic in Section 1
should be re-examined -- it assumes impressions_90d is a stable value proxy.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*  
Per this card's note: the queue CSV stays OUT of git (CI leak-guard blocks
data files) -- the notebook regenerates it on every run. Figures go to
work/figures/ (committed). Metrics go to a JSON in work/outputs/ (committed) --
these are the receipts the paper's numbers trace back to, since the CSV
itself won't be in the repo history.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, json
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# CSV -- gitignored by design, regenerated each run
queue.to_csv("work/outputs/action_queue.csv", index=False)

# Committed metrics JSON -- the receipts
metrics = {
    "rows_scored": int(len(df)),
    "action_mix": df["suggested_action"].value_counts().to_dict(),
    "archetype_table": arch.to_dict(orient="records"),
    "decay_curve": decay[decay["n"] >= 30].to_dict(orient="records"),
    "model": "random_forest_full_data",
    "note": "decline_probability is decision-support risk ranking, not a guarantee",
}
with open("work/outputs/playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2, default=str)

# Simple committed figure: action mix bar chart
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
counts = df["suggested_action"].value_counts()
plt.figure(figsize=(7,4))
counts.plot(kind="barh")
plt.xlabel("Pages")
plt.title("Suggested action mix (30,000 pages scored)")
plt.tight_layout()
plt.savefig("work/figures/action_mix.png", dpi=150)
plt.close()

print("Wrote work/outputs/action_queue.csv (gitignored),",
      "work/outputs/playbook_metrics.json (commit this),",
      "work/figures/action_mix.png (commit this)")

Wrote work/outputs/action_queue.csv (gitignored), work/outputs/playbook_metrics.json (commit this), work/figures/action_mix.png (commit this)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.